In [12]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
import numpy as np


df = pd.read_csv('imdb.csv')
labels = df['sentiment'].map({'positive': 1, 'negative': 0}).values
texts = df['review'].values


In [13]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 64


tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')
y = np.array(labels)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [15]:
def initialize_model():
    model = Sequential([
        Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
        LSTM(64, return_sequences=True),
        LSTM(32),
        Dense(24, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = initialize_model()
model.summary()

c:\Users\darla\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 84s 138ms/step - accuracy: 0.6191 - loss: 0.6364 - val_accuracy: 0.7296 - val_loss: 0.5500
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 104s 178ms/step - accuracy: 0.7680 - loss: 0.5095 - val_accuracy: 0.6031 - val_loss: 0.6923
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 99s 170ms/step - accuracy: 0.7805 - loss: 0.4856 - val_accuracy: 0.6823 - val_loss: 0.5913
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 116s 198ms/step - accuracy: 0.7847 - loss: 0.4605 - val_accuracy: 0.8352 - val_loss: 0.3833
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 99s 170ms/step - accuracy: 0.8626 - loss: 0.3248 - val_accuracy: 0.8442 - val_loss: 0.3606


In [17]:
def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_LEN, padding='post')
    prediction = model.predict(padded_sequence)
    sentiment = "positif" if prediction[0][0] > 0.5 else "négatif"
    confidence = prediction[0][0] if sentiment == "positif" else 1 - prediction[0][0]
    return sentiment, confidence

new_review = "This movie was fantastic! The actors were brilliant and the plot was engaging."
sentiment, confidence = predict_sentiment(new_review)
print(f"Sentiment: {sentiment}, Confiance: {confidence:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step
Sentiment: positif, Confiance: 0.91
